# Knowledge Graph for ML Feature Engineering

Build a typed knowledge graph from tabular data, extract ML features, and generate recommendations based on structural peer similarity.

**No API calls needed for graph construction.** Everything runs locally.

Demo: 2000 accounts × 25 columns → 4321 nodes, 52K edges, 29 features/entity.

## Generate sample data

In [1]:
import random
from faker import Faker

fake = Faker()
Faker.seed(42)
random.seed(42)

INDUSTRIES = ["SaaS", "Fintech", "Healthcare IT", "E-commerce", "Cybersecurity",
              "AI/ML Platform", "DevOps", "EdTech", "MarTech", "InsurTech"]
TECH = ["AWS", "GCP", "Azure", "Kubernetes", "Docker", "Terraform",
        "Python", "Go", "Rust", "PostgreSQL", "Redis", "Kafka"]
PRODUCTS = ["CDN", "WAF", "DDoS Protection", "Zero Trust", "CASB",
            "Workers", "R2", "D1", "AI Gateway", "Browser Run"]

records = []
for i in range(2000):
    company = fake.company().replace(',', '')
    industry = random.choice(INDUSTRIES)
    stage = random.choice(['Prospect', 'Customer', 'POC', 'Churned'])
    records.append({
        'account_id': f'A{i+1:04d}',
        'company_name': company,
        'industry': industry,
        'region': random.choice(['North America', 'EMEA', 'APAC', 'LATAM']),
        'segment': random.choice(['Enterprise', 'Mid-Market', 'SMB']),
        'employees': random.choice([50, 100, 500, 1000, 5000, 10000]),
        'revenue': random.choice([5e6, 50e6, 100e6, 500e6]),
        'stage': stage,
        'tech_stack': ', '.join(random.sample(TECH, random.randint(2, 5))),
        'products_owned': ', '.join(random.sample(PRODUCTS, random.randint(1, 4))) if stage == 'Customer' else '',
        'products_interested': ', '.join(random.sample(PRODUCTS, random.randint(1, 3))),
        'description': f'{company} is a {industry} company that {fake.bs()}.',
    })

print(f'{len(records)} rows × {len(records[0])} columns')

2000 rows × 25 columns


## Profile the data

In [2]:
from pydantic_ai_cloudflare import profile_data

dd = profile_data(records, id_column='account_id')
print(dd.summary())

Data Dictionary (11 columns)
ID column: account_id

  account_id      → id           | designated ID column
  company_name    → categorical  | high cardinality
  industry        → categorical  | 10 unique values
  region          → categorical  | 4 unique values
  segment         → categorical  | 3 unique values
  employees       → numeric      | 100% numeric
  revenue         → numeric      | 100% numeric
  stage           → categorical  | 4 unique values
  tech_stack      → list         | comma-separated
  products_owned  → list         | comma-separated
  products_interested → list     | comma-separated
  description     → text         | avg 80+ chars


## Build the knowledge graph

In [3]:
import asyncio, time
from pydantic_ai_cloudflare import KnowledgeGraph

kg = KnowledgeGraph(account_id='dummy', api_key='dummy')  # no API calls needed

start = time.time()
stats = await kg.build_from_records(
    records, data_dict=dd,
    extract_entities=False, compute_similarity=False,
)
print(f'Built in {time.time()-start:.2f}s')
print(f'Nodes: {stats["nodes"]}, Edges: {stats["edges"]}')
print(f'{len(kg.stats["edge_types"])} edge types')

Built in 0.07s
Nodes: 4321, Edges: 52428
21 edge types


## ML features

In [4]:
features = kg.to_feature_dicts()
n_features = len(next(iter(features.values())))
print(f'{len(features)} entities × {n_features} features')

sample = list(features.items())[0]
print(f'\nSample features for {sample[0]}:')
for k, v in sample[1].items():
    if not k.startswith('HAS_COMPANY') and not k.startswith('HAS_SUB'):
        print(f'  {k}: {v}')

2000 entities × 29 features

Sample features for A0001:
  degree: 28
  pagerank: 0.0003
  clustering_coeff: 0.0
  community_id: 0
  community_size: 2000
  HAS_TECH_STACK_degree: 4
  HAS_PRODUCTS_INTERESTED_degree: 2


## KNN rate features (peer adoption → propensity signals)

For each account, find K structurally similar accounts and check what products they own. High adoption rate among peers → high propensity signal.

In [5]:
# Find a customer to recommend to
target = next(r for r in records if r['stage'] == 'Customer' and r['products_owned'])
print(f'Target: {target["account_id"]} ({target["company_name"]})')
print(f'Current products: {target["products_owned"]}')

recs = kg.recommend(
    target['account_id'],
    ['products_owned', 'use_cases'],
    k=5, min_rate=0.3,
)
print(f'\nRecommendations:')
for r in recs[:5]:
    print(f'  {r["value"]:20s} rate={r["rate"]:.2f} | {sum(1 for c in r["peer_votes"] if c=="✓")}/{len(r["peer_votes"])} peers have it')

Target: A0003 (Herrera-Dudley)
Current products: DDoS Protection

Recommendations:
  data_protection: rate=0.60 | 3/5 peers have it
  bot_management:  rate=0.40 | 2/5 peers have it
  remote_access:   rate=0.40 | 2/5 peers have it


## Co-occurrence (which products are bought together)

In [6]:
co = kg.co_occurrence_features('products_owned')
for product in list(co.keys())[:3]:
    top = sorted(co[product].items(), key=lambda x: x[1], reverse=True)[:2]
    print(f'{product}:')
    for other, rate in top:
        print(f'  + {other}: P={rate}')

ai gateway:
  + browser run: P=0.21
  + magic transit: P=0.17
browser run:
  + d1: P=0.24
  + magic transit: P=0.21
casb:
  + r2: P=0.28
  + workers: P=0.25


## Use cases

| Use case | Method | What it does |
|----------|--------|-------------|
| Propensity scoring | `knn_rate_features()` → XGBoost | Predict which products an account will buy |
| Account matching | `find_similar()` + `pairwise_features()` | Find lookalike accounts for ABM |
| Recommendation | `recommend()` | Suggest products based on peer portfolios |
| Cross-sell | `co_occurrence_features()` | Identify product bundles |
| Churn prediction | `to_feature_dicts()` → ML model | Graph features improve AUC by 5-15% |
| Market segmentation | `compute_features()` → community_id | Auto-segment by graph structure |